# 象热成像 YOLO 模型继续训练

用途：在本机 GPU、实验室服务器或云端 GPU 环境中继续训练当前象群热成像识别模型，并生成可回填到平台网站的训练曲线、混淆矩阵、预测样例和 `best.pt`。

In [ ]:
# 1. 安装依赖。第一次运行会慢一些。
%pip install -r requirements-training.txt

In [ ]:
# 2. 检查 GPU/CUDA 环境。
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 3. 准备公开训练数据。
# 如果你已经用 scripts/package_training.ps1 -IncludeData 打包上传，可以把 DOWNLOAD_PUBLIC_DATASETS 改为 False。
from pathlib import Path
import shutil

DOWNLOAD_PUBLIC_DATASETS = True
downloads = Path('downloads')
downloads.mkdir(exist_ok=True)

def patch_kagglehub_compat():
    import kagglesdk.kaggle_env as kaggle_env
    if not hasattr(kaggle_env, 'get_web_endpoint'):
        kaggle_env.get_web_endpoint = lambda env=None: kaggle_env.get_endpoint(env or kaggle_env.get_env()).replace('api.', 'www.')

def copy_dataset_root(source: Path, target: Path):
    candidates = [source] + [p for p in source.rglob('*') if p.is_dir()]
    root = next((p for p in candidates if (p / 'data.yaml').exists() or (p / 'dataset.yaml').exists()), source)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(root, target)
    print('ready:', target)

if DOWNLOAD_PUBLIC_DATASETS:
    from huggingface_hub import snapshot_download
    patch_kagglehub_compat()
    import kagglehub

    elephant_path = Path(kagglehub.dataset_download('shijo96john/elephant-thermal-images'))
    copy_dataset_root(elephant_path, downloads / 'elephant-thermal-images')

    dogs_path = Path(snapshot_download(repo_id='LibreYOLO/thermal-dogs-and-people-x6ejw', repo_type='dataset', local_dir=str(downloads / 'thermal-dogs-and-people-x6ejw')))
    print('ready:', dogs_path)

    cheetah_path = Path(snapshot_download(repo_id='LibreYOLO/thermal-cheetah-my4dp', repo_type='dataset', local_dir=str(downloads / 'thermal-cheetah-my4dp')))
    print('ready:', cheetah_path)

    hit_path = Path(kagglehub.dataset_download('pandrii000/hituav-a-highaltitude-infrared-thermal-dataset'))
    copy_dataset_root(hit_path, downloads / 'hit-uav-yolo' / 'hit-uav')

    # BAMBI 预训练动物检测权重，小体量，可作为后续迁移训练参考。
    snapshot_download(repo_id='cpraschl/bambi-thermal-detection', local_dir='models/bambi-thermal-detection')

    # BAMBI Detection Dataset 为 9.4GB，如需加入全量训练，取消下面两行注释。
    # !wget -O downloads/bambi-detection-yolov12.zip https://zenodo.org/api/records/15773102/files/bambi-bounding_box-20250523.v1i.yolov12.zip/content
    # !unzip -q downloads/bambi-detection-yolov12.zip -d downloads/bambi-detection-yolov12
else:
    print('Using uploaded local data folders.')

In [ ]:
# 4. 生成多源 YOLO 数据集配置。
!python prepare_elephant_multisource_thermal_yolo.py

In [ ]:
# 5. 继续训练象模型。
# 默认会优先使用上传包里的当前 best.pt；找不到时自动使用 yolov8n.pt。
!python train_elephant_yolo.py --data data/elephant_multisource_thermal.yaml --epochs 30 --imgsz 640 --batch 16 --device auto --workers 2 --project runs/elephant_multisource_thermal --name continue_elephant_yolo --predict --zip

In [ ]:
# 6. 查看训练结果。下载 zip 后，可把 results.png、confusion_matrix.png、labels.jpg、val_batch0_pred.jpg 回填到网站 public/training-results/elephant/。
from pathlib import Path
from IPython.display import Image, display

matches = sorted(Path('runs').rglob('continue_elephant_yolo'))
run_dir = matches[-1] if matches else Path('runs/elephant_multisource_thermal/continue_elephant_yolo')
print('run_dir:', run_dir)
for name in ['results.png', 'confusion_matrix.png', 'labels.jpg', 'val_batch0_pred.jpg']:
    path = run_dir / name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

print('best.pt:', run_dir / 'weights' / 'best.pt')
print('artifact zip:', run_dir.parent / 'continue_elephant_yolo_artifact.zip')